In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

DATASET =  "gencode.v47.common.cdhit.cv"
BASEDIR = Path("/mnt/cbib/LNClassifier/paper/")

sys.path.insert(0, str(BASEDIR / "workflow"))
from utils.features import filter_feature_columns, remove_constant_features
from utils.parsing import load_fasta

fold_datasets = BASEDIR / "results" / DATASET / "datasets"

#### Common transcripts between v46 and v47

In [ ]:
common_gencode = BASEDIR / "results/gencode_comparison/v46_vs_v47/gencode.v47.common_same_class_transcripts.fa"
! grep '^>' {common_gencode} -c

171512


#### Number of transcript clusters / representative sequences / common-CDHIT dataset

In [ ]:
! grep '^>' {fold_datasets / "cdhit" / "*.fa"} -c

112264


All these transcripts were used for training/testing, but not all had full classifications

In [ ]:
full_table = pd.read_csv(BASEDIR / "results/gencode.v47.common.cdhit.cv/tables/gencode.v47.common.cdhit.cv_full_table.tsv", sep="\t")
transcripts_with_features = full_table.shape[0]
transcripts_with_features

112264

Transcripts classified by all models

In [ ]:
binary_table = pd.read_csv(BASEDIR / "results/gencode.v47.common.cdhit.cv/tables/gencode.v47.common.cdhit.cv_binary_class_table.tsv", sep="\t")
classified_transcripts = binary_table.shape[0]
classified_transcripts

111652

In [ ]:
binary_table["real"].value_counts()


real
True     66376
False    45276
Name: count, dtype: int64

In [ ]:
binary_table["real"].value_counts() / classified_transcripts * 100

real
True     59.449002
False    40.550998
Name: count, dtype: float64

In [ ]:
transcripts_without_class = transcripts_with_features - classified_transcripts
transcripts_without_class

612

In [ ]:
tools = ['label_rnasamba',
 'label_feelnc',
 'label_l_cpat',
 'label_ss_lncDC',
 'label_mrnn',
 'label_lncrnabert',
 'label_plncpro',
 'label_ss_lncfinder',]

In [ ]:
full_table[tools].isna().sum()

label_rnasamba          0
label_feelnc            5
label_l_cpat          180
label_ss_lncDC        529
label_mrnn              0
label_lncrnabert        0
label_plncpro           0
label_ss_lncfinder     15
dtype: int64

### Transcript length

In [ ]:
binary_table

,seq_ID,fold,rnasamba,feelnc,p_cpat,l_cpat,lncDC,ss_lncDC,mrnn,lncrnabert,plncpro,ss_lncfinder,lncfinder,real
0,ENST00000000412.8,fold1,True,True,True,True,True,True,True,True,True,True,True,True
1,ENST00000002596.6,fold1,True,True,True,True,True,True,True,True,True,True,True,True
2,ENST00000002829.8,fold1,True,True,True,True,True,True,True,True,True,True,True,True
3,ENST00000005260.9,fold1,True,True,True,True,True,True,True,True,True,True,True,True
4,ENST00000005995.8,fold1,True,True,True,True,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111647,ENST00000715715.1,fold5,False,False,True,False,False,False,False,False,False,False,False,False
111648,ENST00000715718.1,fold5,False,False,True,False,False,False,False,False,False,False,False,False
111649,ENST00000715720.1,fold5,False,False,True,False,False,False,False,False,False,False,False,False
111650,ENST00000715721.1,fold5,False,False,True,True,False,False,False,False,False,False,False,False


In [ ]:
lengths_file = BASEDIR / "resources" / "gencode.v47.transcripts.length.tsv"
lengths = pd.read_csv(lengths_file, sep="\t", index_col=0, header=None, names=["length"])
lengths["real"] = binary_table.set_index("seq_ID")["real"]
lengths = lengths.dropna(subset=["real"])
summary = lengths.groupby("real")["length"].agg(
    mean="mean",
    sd="std",
    median="median",
    q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75),
)

summary["mean ± sd"] = summary.apply(lambda r: f'{r["mean"]:.1f} ± {r["sd"]:.1f}', axis=1)
summary["median (IQR)"] = summary.apply(lambda r: f'{r["median"]:.1f} ({r["q1"]:.1f}–{r["q3"]:.1f})', axis=1)



print(summary[["mean ± sd", "median (IQR)"]].to_string())

             mean ± sd           median (IQR)
real                                         
False  1362.8 ± 1125.3  1028.0 (642.0–1698.0)
True   2352.5 ± 2269.2  1620.0 (726.0–3155.0)


Now let's look at transcript length but using all pc and lncRNA transcripts

In [ ]:
pc_file = BASEDIR / "resources" / "gencode.v47.pc_transcripts.fa"
lnc_file = BASEDIR / "resources" / "gencode.v47.lncRNA_transcripts.fa"

pc_tx = load_fasta(pc_file)
lnc_tx = load_fasta(lnc_file)

In [ ]:
# Calculate lengths for pc and lnc transcripts
pc_lengths = [len(record.seq) for record in pc_tx]
lnc_lengths = [len(record.seq) for record in lnc_tx]

# Create DataFrame for analysis
all_lengths = pd.DataFrame({
    'length': pc_lengths + lnc_lengths,
    'type': ['pc'] * len(pc_lengths) + ['lnc'] * len(lnc_lengths)
})

# Calculate summary statistics
summary_all = all_lengths.groupby('type')['length'].agg(
    count='count',
    mean='mean',
    sd='std',
    median='median',
    q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75),
)

summary_all['mean ± sd'] = summary_all.apply(lambda r: f'{r["mean"]:.1f} ± {r["sd"]:.1f}', axis=1)
summary_all['median (IQR)'] = summary_all.apply(lambda r: f'{r["median"]:.1f} ({r["q1"]:.1f}–{r["q3"]:.1f})', axis=1)

print(f"Protein-coding transcripts: {len(pc_tx)}")
print(f"lncRNA transcripts: {len(lnc_tx)}")
print(f"\n{summary_all[['count', 'mean ± sd', 'median (IQR)']].to_string()}")

Protein-coding transcripts: 112218
lncRNA transcripts: 191106

       count        mean ± sd           median (IQR)
type                                                
lnc   191106  1024.2 ± 1224.0   814.0 (571.0–1224.0)
pc    112218  2425.7 ± 2500.3  1768.5 (772.0–3231.0)


## Number of features

In [ ]:
features_to_keep = filter_feature_columns(full_table)

Identified length columns to exclude: ['Transcript_length_lncDC', 'length_plncpro'] (keeping RNA_size_feelnc for reference)
Total number of columns in features table: 173
Number of kept feature columns: 128
Feature columns: ['kmerScore_1mer_feelnc', 'kmerScore_2mer_feelnc', 'kmerScore_3mer_feelnc', 'kmerScore_6mer_feelnc', 'kmerScore_9mer_feelnc', 'kmerScore_12mer_feelnc', 'ORF_cover_feelnc', 'RNA_size_feelnc', 'ORF_l_cpat', 'Fickett_l_cpat', 'Hexamer_l_cpat', 'ORF_coverage_l_cpat', 'GC_content_lncDC', 'Fickett_score_lncDC', 'ORF_T0_length_lncDC', 'ORF_T1_length_lncDC', 'ORF_T2_length_lncDC', 'ORF_T0_coverage_lncDC', 'ORF_T1_coverage_lncDC', 'ORF_T3_coverage_lncDC', 'Hexamer_score_ORF_T0_lncDC', 'Hexamer_score_ORF_T1_lncDC', 'Hexamer_score_ORF_T2_lncDC', 'Hexamer_score_ORF_T3_lncDC', 'RCB_T0_lncDC', 'RCB_T1_lncDC', 'ORF_T0_PI_lncDC', 'ORF_T0_MW_lncDC', 'ORF_T0_aromaticity_lncDC', 'ORF_T0_instability_lncDC', 'ORF_T1_MW_lncDC', 'ORF_T1_instability_lncDC', 'ORF_T2_MW_lncDC', 'ORF_T3_MW_ln

In [ ]:
full_table[features_to_keep].columns.str.split("_").str[-1].value_counts()

plncpro      69
lncDC        28
lncfinder    19
feelnc        8
cpat          4
Name: count, dtype: int64

In [ ]:
# Summary of features used
selected_features = remove_constant_features(full_table[features_to_keep])
selected_features.columns.str.split("_").str[-1].value_counts()

Dataset: No constant features were removed


plncpro      69
lncDC        28
lncfinder    19
feelnc        8
cpat          4
Name: count, dtype: int64

### TE and NBD pipelines

In [ ]:
te_pipeline = 'te_pipeline/results/te_analysis_flexible/features/all_transcripts_te_features.corrected.csv'
nbd_pipeline = 'nonb-pipeline/results/gencode.v47.transcripts/extended_analysis/features_nonb_features.csv'

te_df = pd.read_csv(BASEDIR / te_pipeline)
nbd_df = pd.read_csv(BASEDIR / nbd_pipeline)
nbd_df.rename(columns={'motif_types_present': 'n_motif_types'}, inplace=True)  # Name change to avoid confusing as constant

In [ ]:
te_exclude_cols = ['transcript_id', 'transcript_length', 'transcript_type', 'coding_class']
te_feature_cols = [c for c in te_df.columns if c not in te_exclude_cols]
display(te_feature_cols)
print(f"Number of TE features: {len(te_feature_cols)}")

['te_min_sw_score',
 'te_max_sw_score',
 'te_mean_sw_score',
 'te_min_divergence',
 'te_max_divergence',
 'te_mean_divergence',
 'te_min_perc_del',
 'te_max_perc_del',
 'te_mean_perc_del',
 'te_min_perc_ins',
 'te_max_perc_ins',
 'te_mean_perc_ins',
 'te_count',
 'te_sum_hit_length',
 'te_min_hit_length',
 'te_mean_hit_length',
 'te_max_hit_length',
 'te_min_hit_reference_coverage',
 'te_mean_hit_reference_coverage',
 'te_max_hit_reference_coverage',
 'te_sum_num_fragments',
 'te_mean_num_fragments',
 'te_max_num_fragments',
 'te_sum_fragmented',
 'te_unique_subfamilies',
 'te_unique_classes',
 'te_unique_families',
 'te_has_dna',
 'te_dna_count',
 'te_has_ervk',
 'te_ervk_count',
 'te_has_ervl',
 'te_ervl_count',
 'te_has_ervl-malr',
 'te_ervl-malr_count',
 'te_has_erv1',
 'te_erv1_count',
 'te_has_line',
 'te_line_count',
 'te_has_ltr',
 'te_ltr_count',
 'te_has_ple',
 'te_ple_count',
 'te_has_rc',
 'te_rc_count',
 'te_has_retroposon',
 'te_retroposon_count',
 'te_has_sine',
 'te_sin

Number of TE features: 170


In [ ]:
te_selected_df = remove_constant_features(te_df[te_feature_cols])
display(te_selected_df.columns.str.split("_").str[0].value_counts())
print(f"Number of selected TE features: {te_selected_df.shape[1]}")

Dataset: Removing 13 constant features
  Constant features: ['te_has_ervk', 'te_ervk_count', 'te_has_ervl', 'te_ervl_count', 'te_has_ervl-malr', 'te_ervl-malr_count', 'te_has_erv1', 'te_erv1_count', 'pseudo_unique_families', 'te_ervk_count_per_kb', 'te_ervl_count_per_kb', 'te_ervl-malr_count_per_kb', 'te_erv1_count_per_kb']


te            69
pseudo        41
lctr          21
unknown       12
global        10
pseudogene     4
Name: count, dtype: int64

Number of selected TE features: 157


### NBD Features

In [ ]:
nbd_exclude_cols = ['transcript_id', 'transcript_length', 'transcript_type', 'coding_class']
nbd_exclude_cols += [c for c in nbd_df.columns if c.endswith("_transcript_id")]
nbd_feature_cols = [c for c in nbd_df.columns if c not in nbd_exclude_cols]
display(nbd_feature_cols)
print(f"Number of nonb features: {len(nbd_feature_cols)}")

['apr_hit_count',
 'apr_total_length',
 'apr_max_length',
 'apr_mean_length',
 'apr_std_length',
 'apr_unique_length',
 'apr_gaps_mean',
 'apr_gaps_median',
 'apr_gaps_max',
 'apr_gaps_min',
 'apr_present',
 'apr_total_length_pct',
 'apr_max_length_pct',
 'apr_mean_length_pct',
 'apr_std_length_pct',
 'apr_unique_length_pct',
 'apr_gaps_mean_pct',
 'apr_gaps_median_pct',
 'apr_gaps_max_pct',
 'apr_gaps_min_pct',
 'apr_hit_count_per_kb',
 'dr_hit_count',
 'dr_total_length',
 'dr_max_length',
 'dr_mean_length',
 'dr_std_length',
 'dr_unique_length',
 'dr_gaps_mean',
 'dr_gaps_median',
 'dr_gaps_max',
 'dr_gaps_min',
 'dr_present',
 'dr_total_length_pct',
 'dr_max_length_pct',
 'dr_mean_length_pct',
 'dr_std_length_pct',
 'dr_unique_length_pct',
 'dr_gaps_mean_pct',
 'dr_gaps_median_pct',
 'dr_gaps_max_pct',
 'dr_gaps_min_pct',
 'dr_hit_count_per_kb',
 'gq_hit_count',
 'gq_total_length',
 'gq_max_length',
 'gq_mean_length',
 'gq_std_length',
 'gq_unique_length',
 'gq_gaps_mean',
 'gq_gaps

Number of nonb features: 178


In [ ]:
len(['apr_hit_count',
 'apr_total_length',
 'apr_max_length',
 'apr_mean_length',
 'apr_std_length',
 'apr_unique_length',
 'apr_gaps_mean',
 'apr_gaps_median',
 'apr_gaps_max',
 'apr_gaps_min',
 'apr_present',
 'apr_total_length_pct',
 'apr_max_length_pct',
 'apr_mean_length_pct',
 'apr_std_length_pct',
 'apr_unique_length_pct',
 'apr_gaps_mean_pct',
 'apr_gaps_median_pct',
 'apr_gaps_max_pct',
 'apr_gaps_min_pct',
 'apr_hit_count_per_kb',]) * 8
 

168

In [ ]:
nbd_selected_df = remove_constant_features(nbd_df[nbd_feature_cols])
display(nbd_selected_df.columns.str.split("_").str[0].value_counts())
print(f"Number of selected TE features: {nbd_selected_df.shape[1]}")

Dataset: No constant features were removed


apr      21
dr       21
gq       21
ir       21
mr       21
str      21
tri      21
z        21
all       4
total     3
any       1
n         1
motif     1
Name: count, dtype: int64

Number of selected TE features: 178


In [ ]:
# Separate continuous (numeric) from categorical features
for df in [selected_features, te_selected_df, nbd_selected_df]:
    print(f"{df.shape[1]} features in {df.shape[0]} transcripts")
    test_df = df.copy()
    categorical_features = []
    continuous_features = []

    cat_substrings = ['_has_', '_present']
    not_cat = ["motif_types_present"]

    cat_cols = [col for col in test_df.columns if any(sub in col for sub in cat_substrings) and col not in not_cat]
    # Remaining are continuous (numeric)
    continuous_cols = test_df.columns.difference(cat_cols).tolist()

    if cat_cols:
        categorical_features = test_df[cat_cols]
    if continuous_cols:
        continuous_features = test_df[continuous_cols]

    # Check for duplicate values in categorical columns
    n_cat_constant = test_df[cat_cols].nunique().value_counts().get(1, 0)

    print(f"  Categorical features: {len(cat_cols)} (containing substrings {cat_substrings})")
    print(f"  Categorical columns: {cat_cols}")
    print(f"  Constant categorical features: {n_cat_constant}")
    print(f"  Continuous features: {len(continuous_cols)}")
    if 'transcript_length' in continuous_cols:
        print("  Warning: 'transcript_length' is present in dataset")

128 features in 112264 transcripts
  Categorical features: 0 (containing substrings ['_has_', '_present'])
  Categorical columns: []
  Constant categorical features: 0
  Continuous features: 128
157 features in 385659 transcripts
  Categorical features: 15 (containing substrings ['_has_', '_present'])
  Categorical columns: ['te_has_dna', 'te_has_line', 'te_has_ltr', 'te_has_ple', 'te_has_rc', 'te_has_retroposon', 'te_has_sine', 'te_has_srprna', 'lctr_has_low_complexity', 'lctr_has_simple_repeat', 'lctr_has_satellite', 'pseudo_has_rrna', 'pseudo_has_scrna', 'pseudo_has_snrna', 'pseudo_has_trna']
  Constant categorical features: 0
  Continuous features: 142
178 features in 385659 transcripts
  Categorical features: 9 (containing substrings ['_has_', '_present'])
  Categorical columns: ['apr_present', 'dr_present', 'gq_present', 'ir_present', 'mr_present', 'str_present', 'tri_present', 'z_present', 'any_nonb_present']
  Constant categorical features: 0
  Continuous features: 169


In [ ]:
SHAP_DIR: Path = Path(
    "/mnt/cbib/LNClassifier/paper/results"
    "/gencode.v47.common.cdhit.cv/features/shap_clustered"
)
per_fold_mean_abs = pd.read_csv(SHAP_DIR / "shap_per_fold_mean_abs.csv", index_col=0)
per_fold_mean_abs.columns

Index(['te_has_dna', 'te_has_line', 'te_has_ltr', 'te_has_ple', 'te_has_rc',
       'te_has_retroposon', 'te_has_sine', 'te_has_srprna',
       'lctr_has_low_complexity', 'lctr_has_simple_repeat',
       ...
       'tri_hit_count', 'tri_hit_count_per_kb', 'tri_std_length',
       'unknown_count', 'z_gaps_max_pct', 'z_gaps_mean', 'z_gaps_min',
       'z_hit_count', 'z_hit_count_per_kb', 'z_std_length'],
      dtype='object', length=179)

### Replicate how the feature clustering notebook loads features

In [ ]:
from utils.entropy import load_additional_features, load_dataset
DATASET_NAME = "gencode.v47.common.cdhit.cv"

In [ ]:
# Load main dataset
dataset = load_dataset(DATASET_NAME)
dataset.update(load_additional_features(DATASET_NAME, BASEDIR))

probs = dataset['probs']
labels = dataset['labels']
features = dataset['features']
features_to_keep = filter_feature_columns(features)
features = features[features_to_keep]
binary = dataset['binary']

# Load additional features if available
if 'te_pipeline' in dataset:
    te_features = dataset['te_pipeline'].loc[probs.index]
    te_features = te_features.fillna(0)
else:
    te_features = None
    print("⚠ TE features not loaded")

if 'nbd_pipeline' in dataset:
    nbd_features = dataset['nbd_pipeline'].loc[probs.index]
    nbd_features.rename(columns={'motif_types_present': 'n_motif_types'}, inplace=True)
else:
    nbd_features = None
    print("⚠ NBD features not loaded")

if 'entropy' in dataset:
    entropy_df = dataset['entropy'].loc[probs.index]
else:
    entropy_df = None
    print("⚠ Entropy metrics not loaded")

print(f"✓ Data loaded: {len(probs)} transcripts with {len(features.columns)} features")

Extracted 8 probability columns.
Inverting noncoding probabilities...
  - Inverting column: Noncoding_prob_ss_lncDC
✓ Loaded te_pipeline (385659 samples × 173 features)
✓ Loaded nbd_pipeline (385659 samples × 189 features)
✓ Loaded entropy (111652 samples × 13 features)
Identified length columns to exclude: ['Transcript_length_lncDC', 'length_plncpro'] (keeping RNA_size_feelnc for reference)
Total number of columns in features table: 172
Number of kept feature columns: 128
Feature columns: ['kmerScore_1mer_feelnc', 'kmerScore_2mer_feelnc', 'kmerScore_3mer_feelnc', 'kmerScore_6mer_feelnc', 'kmerScore_9mer_feelnc', 'kmerScore_12mer_feelnc', 'ORF_cover_feelnc', 'RNA_size_feelnc', 'ORF_l_cpat', 'Fickett_l_cpat', 'Hexamer_l_cpat', 'ORF_coverage_l_cpat', 'GC_content_lncDC', 'Fickett_score_lncDC', 'ORF_T0_length_lncDC', 'ORF_T1_length_lncDC', 'ORF_T2_length_lncDC', 'ORF_T0_coverage_lncDC', 'ORF_T1_coverage_lncDC', 'ORF_T3_coverage_lncDC', 'Hexamer_score_ORF_T0_lncDC', 'Hexamer_score_ORF_T1_ln

In [ ]:
original_full = pd.concat([features, te_features, nbd_features], axis=1)
full_feature_set = original_full.copy()
# Since merged df may have duplicate columns:
full_feature_set = full_feature_set.loc[:, ~full_feature_set.columns.duplicated(keep='first')]
# Fill missing values (possible for categorical TE features)
full_feature_set.fillna(0, inplace=True)
# Convert boolean/object columns to numerical (0/1)
full_feature_set = full_feature_set.apply(pd.to_numeric, errors='coerce')

# Keep only numeric features, and report and remove constant features
numeric_cols = full_feature_set.select_dtypes(include=[np.number]).columns
print(f"Full feature set: keeping {len(numeric_cols)} numeric features (out of {full_feature_set.shape[1]} total)")
numeric_df = full_feature_set[numeric_cols]
nunique = numeric_df.nunique()
constant_features = nunique[nunique <= 1].index.tolist()
if constant_features:
    print(f"Full feature set: Removing {len(constant_features)} constant features")
    print(f"  Constant features: {constant_features}")
    full_feature_set = numeric_df.drop(columns=constant_features)

Full feature set: keeping 487 numeric features (out of 487 total)
Full feature set: Removing 23 constant features
  Constant features: ['te_has_ervk', 'te_ervk_count', 'te_has_ervl', 'te_ervl_count', 'te_has_ervl-malr', 'te_ervl-malr_count', 'te_has_erv1', 'te_erv1_count', 'pseudo_unique_families', 'te_ervk_count_per_kb', 'te_ervl_count_per_kb', 'te_ervl-malr_count_per_kb', 'te_erv1_count_per_kb', 'transcript_type', 'coding_class', 'apr_transcript_id', 'dr_transcript_id', 'gq_transcript_id', 'ir_transcript_id', 'mr_transcript_id', 'str_transcript_id', 'tri_transcript_id', 'z_transcript_id']


In [ ]:
for df, df_name, other_df in zip([features, te_features, nbd_features],
                                ["ML features", "TE features", "NBD features"],
                                [selected_features, te_selected_df, nbd_selected_df]):
    

    test_df = df.copy()
    # 1. Only numerical features
    numeric_df = test_df[test_df.select_dtypes(include=[np.number]).columns]
    
    # 2. Only non-constant features
    nunique = numeric_df.nunique()
    constant_features = nunique[nunique <= 1].index.tolist()
    not_constant_features = nunique[nunique > 1].index.tolist()
    test_df = numeric_df.drop(columns=constant_features)

    # 3. Separate continuous from categorical features
    categorical_features = []
    continuous_features = []

    cat_substrings = ['_has_', '_present']

    cat_cols = [col for col in test_df.columns if any(sub in col for sub in cat_substrings)]
    continuous_cols = test_df.columns.difference(cat_cols).tolist()

    not_common = set(df.columns) - set(other_df.columns)

    print(f"{df_name}: {df.shape[1]} features in {df.shape[0]} transcripts")
    print(f"  Numeric features: {numeric_df.shape[1]}")
    print(f"  Removing {len(constant_features)} constant features")
    print(f"  Keeping {len(not_constant_features)} variable features")
    print(f"  Categorical features: {len(cat_cols)} (containing substrings {cat_substrings})")
    print(f"  Continuous features: {len(continuous_cols)}")
    print(f"  DF {'has' if 'transcript_length' in test_df.columns else 'does not have'} transcript_length feature")
    print(f"  Continuous features present in full feature set: {len([c for c in continuous_cols if c in full_feature_set.columns])}")
    print(f"  Categorical features present in full feature set: {len([c for c in cat_cols if c in full_feature_set.columns])}")
    print(f"  Features in this set but not in full feature set: {len(not_common)}")
    print(f"    {list(not_common)[:10]}")
    print()

ML features: 128 features in 111652 transcripts
  Numeric features: 128
  Removing 0 constant features
  Keeping 128 variable features
  Categorical features: 0 (containing substrings ['_has_', '_present'])
  Continuous features: 128
  DF does not have transcript_length feature
  Continuous features present in full feature set: 128
  Categorical features present in full feature set: 0
  Features in this set but not in full feature set: 0
    []

TE features: 173 features in 111652 transcripts
  Numeric features: 152
  Removing 9 constant features
  Keeping 143 variable features
  Categorical features: 0 (containing substrings ['_has_', '_present'])
  Continuous features: 143
  DF has transcript_length feature
  Continuous features present in full feature set: 143
  Categorical features present in full feature set: 0
  Features in this set but not in full feature set: 16
    ['te_has_ervl-malr', 'te_ervl_count_per_kb', 'te_has_ervk', 'te_ervk_count', 'transcript_length', 'coding_class',

### ML Features

In [ ]:
test_df = features.copy()
df_name = "ML set"

print(f"Analyzing set: {df_name}")

# 1. Only numerical features
numeric_df = test_df[test_df.select_dtypes(include=[np.number]).columns]

# 2. Only non-constant features
nunique = numeric_df.nunique()
constant_features = nunique[nunique <= 1].index.tolist()
not_constant_features = nunique[nunique > 1].index.tolist()
test_df = numeric_df.drop(columns=constant_features)

# 3. Separate continuous from categorical features
categorical_features = []
continuous_features = []

cat_substrings = ['_has_', '_present']
cat_cols = [col for col in test_df.columns if any(sub in col for sub in cat_substrings)]
continuous_cols = test_df.columns.difference(cat_cols).tolist()

print(f"{df_name}: {df.shape[1]} features in {df.shape[0]} transcripts")
print(f"  Numeric features: {numeric_df.shape[1]}")
print(f"  Removing {len(constant_features)} constant features")
print(f"  Keeping {len(not_constant_features)} variable features")
print(f"  Categorical features: {len(cat_cols)}")
print(f"  Continuous features: {len(continuous_cols)}")
print()

Analyzing set: ML set
ML set: 189 features in 111652 transcripts
  Numeric features: 128
  Removing 0 constant features
  Keeping 128 variable features
  Categorical features: 0
  Continuous features: 128



In [ ]:
test_df

,kmerScore_1mer_feelnc,kmerScore_2mer_feelnc,kmerScore_3mer_feelnc,kmerScore_6mer_feelnc,kmerScore_9mer_feelnc,kmerScore_12mer_feelnc,ORF_cover_feelnc,RNA_size_feelnc,ORF_l_cpat,Fickett_l_cpat,...,Signal.Q2_lncfinder,Signal.Max_lncfinder,Dot_lnc.dist_lncfinder,Dot_pct.dist_lncfinder,Dot_Dist.Ratio_lncfinder,SS.lnc.dist_lncfinder,SS.pct.dist_lncfinder,SS.Dist.Ratio_lncfinder,MFE_lncfinder,UP.PCT_lncfinder
seq_ID,,,,,,,,,,,,,,,,,,,,,
ENST00000000412.8,0.500532,0.500517,0.521942,0.547063,0.550243,0.484597,0.340408,2450.0,834.0,0.9882,...,3.612083,9.503012,-0.035259,-0.038387,0.918522,-0.041452,-0.043689,0.948800,-766.2,0.109024
ENST00000002596.6,0.506236,0.516955,0.552951,0.614614,0.638448,0.500128,0.129050,7160.0,924.0,1.3522,...,9.930891,34.428530,-0.003786,-0.006761,0.560014,-0.010505,-0.012413,0.846316,-2201.2,0.110211
ENST00000002829.8,0.506685,0.516865,0.550626,0.604411,0.633491,0.513626,0.653729,3607.0,2358.0,1.1447,...,5.809128,14.438914,-0.027893,-0.036709,0.759835,-0.026237,-0.031147,0.842343,-1583.8,0.118414
ENST00000005260.9,0.502833,0.508791,0.530998,0.563220,0.573943,0.468130,0.486420,3645.0,1536.0,0.9867,...,5.441633,20.423523,-0.008949,-0.015182,0.589437,-0.014525,-0.018515,0.784500,-1231.3,0.116905
ENST00000005995.8,0.505395,0.513231,0.533184,0.566367,0.577786,0.493292,0.893676,1091.0,945.0,0.9491,...,1.691524,5.218731,-0.108187,-0.132658,0.815534,-0.097378,-0.110237,0.883350,-440.1,0.122936
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENST00000715715.1,0.498402,0.497692,0.493839,0.487290,0.453654,0.375686,0.187023,1572.0,294.0,0.5278,...,2.273105,7.508407,-0.043092,-0.053699,0.802472,-0.068444,-0.072700,0.941463,-453.3,0.110757
ENST00000715718.1,0.502086,0.505426,0.503883,0.495754,0.449565,0.332127,0.185593,1180.0,219.0,0.6443,...,1.494416,3.475254,-0.096657,-0.100043,0.966157,-0.128784,-0.126897,1.014876,-315.9,0.111111
ENST00000715720.1,0.508883,0.522687,0.527667,0.548416,0.509664,0.312139,0.089971,4168.0,264.0,0.5106,...,5.608470,23.138216,-0.009223,-0.012621,0.730803,-0.016983,-0.020005,0.848949,-1340.7,0.111831


In [ ]:
test_df = te_features.copy()
df_name = "TE set"
print(f"Analyzing set: {df_name}")

# 0. Preprocessing
# 0.1 Convert boolean/object TE columns to numeric where possible
bool_cols = test_df.select_dtypes(include=["bool"]).columns
obj_cols = test_df.select_dtypes(include=["object"]).columns

if len(bool_cols) > 0:
    test_df[bool_cols] = test_df[bool_cols].astype(int)

if len(obj_cols) > 0:
    test_df[obj_cols] = test_df[obj_cols].apply(pd.to_numeric, errors="ignore")

# 0.2 Remove ids
non_id_cols = [col for col in test_df.columns if not col.endswith("_id")]

# 0.3 Remove metadata
metadata_cols = ["transcript_type", "coding_class", 'transcript_length']
test_df = test_df[non_id_cols].drop(columns=metadata_cols)

# 1. Only numerical features
numeric_df = test_df[test_df.select_dtypes(include=[np.number]).columns]

# 2. Only non-constant features
nunique = numeric_df.nunique()
constant_features = nunique[nunique <= 1].index.tolist()
not_constant_features = nunique[nunique > 1].index.tolist()
# test_df = numeric_df.drop(columns=constant_features)

# 3. Separate continuous from categorical features
categorical_features = []
continuous_features = []

cat_substrings = ['_has_', '_present']
cat_cols = [col for col in test_df.drop(columns=constant_features).columns if any(sub in col for sub in cat_substrings)]
continuous_cols = test_df.drop(columns=constant_features).columns.difference(cat_cols).tolist()

print(f"{df_name}: {test_df.shape[1]} features in {test_df.shape[0]} transcripts")
print(f"  Numeric features: {numeric_df.shape[1]}")
print(f"  Removing {len(constant_features)} constant features")
print(f"  Keeping {len(not_constant_features)} variable features")
print(f"  Categorical features: {len(cat_cols)}")
print(f"  Continuous features: {len(continuous_cols)}")
print()

Analyzing set: TE set
TE set: 170 features in 111652 transcripts
  Numeric features: 170
  Removing 13 constant features
  Keeping 157 variable features
  Categorical features: 15
  Continuous features: 142



In [ ]:
test_df = nbd_features.copy()
df_name = "TE set"
print(f"Analyzing set: {df_name}")

# 0. Preprocessing
# 0.1 Convert boolean/object TE columns to numeric where possible
bool_cols = test_df.select_dtypes(include=["bool"]).columns
obj_cols = test_df.select_dtypes(include=["object"]).columns

if len(bool_cols) > 0:
    test_df[bool_cols] = test_df[bool_cols].astype(int)

if len(obj_cols) > 0:
    test_df[obj_cols] = test_df[obj_cols].apply(pd.to_numeric, errors="ignore")

# 0.2 Remove ids
non_id_cols = [col for col in test_df.columns if not col.endswith("_id")]

# 0.3 Remove metadata
metadata_cols = ["transcript_type", "coding_class", 'transcript_length']
test_df = test_df[non_id_cols].drop(columns=metadata_cols)

# 1. Only numerical features
numeric_df = test_df[test_df.select_dtypes(include=[np.number]).columns]

# 2. Only non-constant features
nunique = numeric_df.nunique()
constant_features = nunique[nunique <= 1].index.tolist()
not_constant_features = nunique[nunique > 1].index.tolist()
# test_df = numeric_df.drop(columns=constant_features)

# 3. Separate continuous from categorical features
categorical_features = []
continuous_features = []

cat_substrings = ['_has_', '_present']
cat_cols = [col for col in test_df.drop(columns=constant_features).columns if any(sub in col for sub in cat_substrings)]
continuous_cols = test_df.drop(columns=constant_features).columns.difference(cat_cols).tolist()

print(f"{df_name}: {test_df.shape[1]} features in {test_df.shape[0]} transcripts")
print(f"  Numeric features: {numeric_df.shape[1]}")
print(f"  Removing {len(constant_features)} constant features")
print(f"  Keeping {len(not_constant_features)} variable features")
print(f"  Categorical features: {len(cat_cols)}")
print(f"  Continuous features: {len(continuous_cols)}")
print()

Analyzing set: TE set


TE set: 178 features in 111652 transcripts
  Numeric features: 178
  Removing 0 constant features
  Keeping 178 variable features
  Categorical features: 9
  Continuous features: 169



In [ ]:
test_df

,apr_hit_count,apr_total_length,apr_max_length,apr_mean_length,apr_std_length,apr_unique_length,apr_gaps_mean,apr_gaps_median,apr_gaps_max,apr_gaps_min,...,all_nonb_gaps_mean,all_nonb_gaps_median,all_nonb_gaps_max,all_nonb_gaps_min,any_nonb_present,n_motif_types,total_nonb_count,total_nonb_coverage,total_nonb_coverage_pct,motif_diversity
seq_ID,,,,,,,,,,,,,,,,,,,,,
ENST00000000412.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,905.200000,232.5,3857.0,8.0,1,3,9.0,224.0,9.142857,8.486856e-01
ENST00000002596.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1347.307692,107.5,28806.0,-23.0,1,6,27.0,708.0,9.888268,1.476695e+00
ENST00000002829.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1439.521739,58.0,15442.0,-53.0,1,5,25.0,655.0,18.159135,1.388193e+00
ENST00000005260.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8400.076923,346.0,80617.0,25.0,1,3,12.0,228.0,6.255144,8.675632e-01
ENST00000005995.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,736.000000,314.0,2750.0,-25.0,1,4,5.0,62.0,5.682860,1.332179e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENST00000715715.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7196.250000,832.0,27052.0,69.0,1,3,3.0,74.0,4.707379,1.098612e+00
ENST00000715718.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,67796.000000,67796.0,111597.0,23995.0,1,1,1.0,16.0,1.355932,-1.000000e-10
ENST00000715720.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8926.100000,110.5,87023.0,-18.0,1,4,22.0,324.0,7.773512,9.828476e-01


### More tests

In [ ]:
dataset['features'][["RNA_size_feelnc", "length_plncpro", "Transcript_length_lncDC"]]

,RNA_size_feelnc,length_plncpro,Transcript_length_lncDC
seq_ID,,,
ENST00000000412.8,2450.0,2450,2450.0
ENST00000002596.6,7160.0,7160,7160.0
ENST00000002829.8,3607.0,3607,3607.0
ENST00000005260.9,3645.0,3645,3645.0
ENST00000005995.8,1091.0,1091,1091.0
...,...,...,...
ENST00000715715.1,1572.0,1572,1572.0
ENST00000715718.1,1180.0,1180,1180.0
ENST00000715720.1,4168.0,4168,4168.0


In [ ]:
awk_transcript_length = pd.read_csv("/mnt/cbib/LNClassifier/paper/nonb-pipeline/resources/transcript_lengths.txt",
index_col=0, names=["transcript_length"], sep="\t")
awk_transcript_length = awk_transcript_length.reindex(dataset['features'].index)

In [ ]:
# Get common transcripts between the two dataframes
common_transcripts = set(awk_transcript_length.index).intersection(set(dataset['features'].index))

# Filter both dataframes to only common transcripts
awk_common = awk_transcript_length.loc[list(common_transcripts)]
features_common = dataset['features'].loc[list(common_transcripts)]

# Extract transcript length from features (using one of the available length columns)
# Since there are multiple length-related columns, compare with the awk version
comparison = pd.DataFrame({
    'awk_length': awk_common['transcript_length'],
    'features_RNA_size': features_common['RNA_size_feelnc'],
    'features_length_plncpro': features_common['length_plncpro'],
    'features_Transcript_length_lncDC': features_common['Transcript_length_lncDC']
})

# Check if all are equal
all_equal = comparison.apply(lambda row: len(set(row.dropna())) <= 1, axis=1)
print(f"Transcripts with matching lengths: {all_equal.sum()} / {len(all_equal)}")
print(f"\nMismatches: {(~all_equal).sum()}")
comparison[~all_equal].head(10)

Transcripts with matching lengths: 111652 / 111652

Mismatches: 0


,awk_length,features_RNA_size,features_length_plncpro,features_Transcript_length_lncDC
seq_ID,,,,


In [ ]:
# Present in correlation - present in this notebook
set(te_features.columns).intersection(full_feature_set) - set(te_selected_df.columns)

{'transcript_length'}

In [ ]:
# Present in correlation - present in this notebook
set(nbd_features.columns).intersection(full_feature_set) - set(nbd_selected_df.columns)

{'transcript_length'}

In [ ]:
# Present in correlation - present in this notebook
set(te_features.columns) - set(te_selected_df.columns)

{'coding_class',
 'pseudo_unique_families',
 'te_erv1_count',
 'te_erv1_count_per_kb',
 'te_ervk_count',
 'te_ervk_count_per_kb',
 'te_ervl-malr_count',
 'te_ervl-malr_count_per_kb',
 'te_ervl_count',
 'te_ervl_count_per_kb',
 'te_has_erv1',
 'te_has_ervk',
 'te_has_ervl',
 'te_has_ervl-malr',
 'transcript_length',
 'transcript_type'}

### Entropy data

In [ ]:
dataset_name = DATASET
entropy_file = f"/mnt/cbib/LNClassifier/paper/results/{dataset_name}/features/entropy/{dataset_name}_uncertainty_analysis.tsv"
entropy_df = pd.read_csv(entropy_file, sep="\t")

low_perc_th = 0.1
high_perc_th = 0.9
low_th = np.percentile(entropy_df['H_pred'], low_perc_th * 100)
high_th = np.percentile(entropy_df['H_pred'], high_perc_th * 100)
high_th_bald = np.percentile(entropy_df['I_bald'], high_perc_th * 100)

low_mask = entropy_df['H_pred'] <= low_th
high_mask = (entropy_df['H_pred'] >= high_th) & (entropy_df['I_bald'] >= high_th_bald)

low_tx = entropy_df[low_mask]
high_tx = entropy_df[high_mask]

pc_low = low_tx[low_tx["true_label"] == 1]
pc_high = high_tx[high_tx["true_label"] == 1]

lnc_low = low_tx[low_tx["true_label"] == 0]
lnc_high = high_tx[high_tx["true_label"] == 0]

print("Transcripts in low entropy group:")
print(f"  - Total: {len(low_tx)}")
print(f"  - pc: {len(pc_low)}")
print(f"  - lnc: {len(lnc_low)}")

print("Transcripts in high entropy group:")
print(f"  - Total: {len(high_tx)}")
print(f"  - pc: {len(pc_high)}")
print(f"  - lnc: {len(lnc_high)}")


KeyError: 'true_label'

### Clusters

In [ ]:
cluster_file = "/mnt/cbib/LNClassifier/paper/results/gencode.v47.common.cdhit.cv/features/clustering/feature_clusters_at_distances.csv"
cluster_df = pd.read_csv(cluster_file, index_col=0)
cluster_df["feature_origin"] = "models"

te_check_cols = [c for c in te_feature_cols if c in cluster_df.index]

nbd_check_cols = [c for c in nbd_feature_cols if c in cluster_df.index]
cluster_df.loc[te_check_cols, "feature_origin"] = "TE"
cluster_df.loc[nbd_check_cols, "feature_origin"] = "NBD"
cluster_df

,cluster_0.05,cluster_0.10,cluster_0.15,cluster_0.20,cluster_0.25,cluster_0.30,cluster_0.35,cluster_0.40,cluster_0.45,cluster_0.50,...,cluster_1.05,cluster_1.10,cluster_1.15,cluster_1.20,cluster_1.25,cluster_1.30,cluster_1.35,cluster_1.40,cluster_1.45,feature_origin
feature,,,,,,,,,,,,,,,,,,,,,
AAA_plncpro,90,83,75,72,67,64,56,50,44,34,...,12,9,8,7,7,6,5,5,5,models
AAC_plncpro,66,59,53,51,48,46,42,38,33,27,...,10,7,6,5,5,4,4,4,4,models
AAG_plncpro,63,56,50,48,46,44,41,37,32,26,...,10,7,6,5,5,4,4,4,4,models
AAT_plncpro,88,81,74,71,67,64,56,50,44,34,...,12,9,8,7,7,6,5,5,5,models
ACA_plncpro,68,61,55,53,50,48,44,39,34,27,...,10,7,6,5,5,4,4,4,4,models
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
z_std_length_pct,165,150,138,130,124,120,107,97,87,76,...,36,33,31,27,26,24,22,19,15,NBD
z_total_length,167,152,140,131,125,121,108,98,88,77,...,37,33,31,27,26,24,22,19,15,NBD
z_total_length_pct,167,152,140,131,125,121,108,98,88,77,...,37,33,31,27,26,24,22,19,15,NBD


: 

: 

: 

: 

: 

In [ ]:
cluster_df["cluster_0.25"].nunique()

155

: 

: 

: 

: 

: 

Are all clusters mono-feature-type?

In [ ]:
cluster_df.groupby("cluster_0.25")["feature_origin"].nunique().value_counts()

feature_origin
1    155
Name: count, dtype: int64

: 

: 

: 

: 

: 

In [ ]:
cluster_df.groupby("cluster_0.25")["feature_origin"].head(1).value_counts()

feature_origin
models    80
TE        41
NBD       34
Name: count, dtype: int64

: 

: 

: 

: 

: 